In [3]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from hbn.constants import Defaults
from hbn.visualization import visualize

import warnings
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

%matplotlib inline

#pio.renderers.default = 'iframe'
import plotly.io as pio
pio.renderers.default='notebook'

#import ipywidgets as widgets       # interactive display
#%config InlineBackend.figure_format = 'svg' # other available formats are: 'retina', 'png', 'jpeg', 'pdf'

In [1]:
# check models
from hbn.models.predictive_modeling import check_models

check_models(filter='*all_feature_models/*')

/Users/maedbhking/Documents/healthy_brain_network/hbn/data/make_dataset.py:18: DtypeWarning: Columns (150) have mixed types. Specify dtype option on import or set low_memory=False.
  dx = pd.read_csv(dx_file)


2023-03-16_15-03-36-30: ['No Diagnosis Given', 'ADHD-Combined Type', 'ADHD-Inattentive Type', 'Other Specified Attention-Deficit/Hyperactivity Disorder', 'ADHD-Hyperactive/Impulsive Type', 'Unspecified Attention-Deficit/Hyperactivity Disorder']: ['DX_01_Cat_new_binarize']: ['female', 'male']
2023-03-19_15-30-25-25S-53: ['ADHD-Combined Type', 'ADHD-Inattentive Type', 'No Diagnosis Given', 'Other Specified Attention-Deficit/Hyperactivity Disorder', 'ADHD-Hyperactive/Impulsive Type', 'Unspecified Attention-Deficit/Hyperactivity Disorder']: ['DX_01_Cat_new_binarize']: ['male']
2023-03-21_17-42-28-28S-61: ['No Diagnosis Given', 'Autism Spectrum Disorder']: ['DX_01_Cat_new_binarize']: ['female', 'male']
2023-03-19_15-45-21-21S-72: ['No Diagnosis Given', 'Specific Learning Disorder with Impairment in Reading']: ['DX_01_Cat_new_binarize']: ['female', 'male']
2023-03-17_16-20-51-94: ['No Diagnosis Given', 'ADHD-Combined Type', 'ADHD-Inattentive Type', 'Other Specified Attention-Deficit/Hyperact

In [9]:

# 2023 models
models_dict = {
             #'Anxiety': 'all_feature_models/2023-03-09_14-35-10-10S-77', 
              #'Depression': 'all_feature_models/2023-03-09_14-35-15-15S-12', 
              #'ADHD': 'all_feature_models/2023-03-09_14-31-02-61',
              #'ASD': 'all_feature_models/2023-03-09_14-35-06-06S-61',
              #'Learning-Disorder-Reading-Impairment': 'all_feature_models/2023-03-09_14-35-19-19S-5',
              #'ASD-females': 'all_feature_models/2023-03-09_14-35-37-37S-20',
              # 'ASD-males': 'all_feature_models/2023-03-09_14-35-31-31S-25',
              # 'ADHD-females': 'all_feature_models/2023-03-09_14-35-27-27S-21',
              # 'ADHD-males': 'all_feature_models/2023-03-09_14-35-23-23S-59'
             }

models_dict = { 'ADHD-incl-comorbidities': 'all_feature_models/2023-03-16_15-03-36-30', # incl comobordities
                'ADHD': 'all_feature_models/2023-03-17_16-20-51-94',
               'ASD': ,
               'Depression', 
               'Reading Impairment'

# loop over models
df_all = pd.DataFrame()
for key, value in models_dict.items():
    MODEL_DIR = os.path.join(Defaults.MODEL_DIR, value)

    df_classify = pd.read_csv(os.path.join(MODEL_DIR, 'classifier-all-phenotypic-models-performance.csv'))
    df_classify['data'] = df_classify['data'].map({'model-data': 'null', 'model-null': 'data'})
    df_classify['participant_group'] = key
    
    df_all = pd.concat([df_all, df_classify])

In [10]:
# set plotting style
visualize.plotting_style()

## Models are trained to classify diagnoses across 5 major disorders: ADHD, ASD, Anxiety, Depression, and Reading Impairment

* Each figure describes a model (Decision Tree Classifier) trained on one measure (e.g., CELF) from a particular domain (e.g., Language Tasks)
* Models (x axis) show ROC AUC results

### Figure Description
* Each figure describes a model (Decision Tree Classifier) trained on one measure (e.g., CELF) from a particular domain (e.g., Language Tasks)
* Models (x axis) show ROC AUC results

### Exceptions:
* Demographics are included in these models: race and ethnicity, sex, age, comborbidities, diagnosis subtype is NOT INCLUDED
* Only numeric variables are included (no categorical variables)
* SMOTE (minor class upsampling) NOT applied to models

## ADHD - difference between including comobidities in a basic demographic model versus excluding it from model
### demographic variables are race, ethnicity, age

In [20]:
df = df_all[df_all['measures']=='Demographics']

df1 = df[(df['target']=='DX_01_Cat_new_binarize')]

visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title='')

## All features - ADHD - Parent vs. Child vs. Teacher


In [18]:
df = df_all[df_all['participant_group']=='ADHD']
df['new_col'] = df['assessment'] + '-' + df['abbrevs']

df1 = df[(df['target']=='DX_01_Cat_new_binarize')]

visualize.predictive_modeling_group(df=df1, x='new_col', y='roc_auc_score', title='')

## All features - ASD - Parent vs. Child vs. Teacher

In [21]:
df = df_all[df_all['participant_group']=='ASD']
df['new_col'] = df['assessment'] + '-' + df['abbrevs']

df1 = df[(df['target']=='DX_01_Cat_new_binarize')]

visualize.predictive_modeling_group(df=df1, x='new_col', y='roc_auc_score', title='')

## Child Measures - Language Tasks

In [13]:
df_all['assessment'].unique()

array(['Teacher Measures', 'Parent Measures', 'Child Measures'],
      dtype=object)

In [17]:
domain = 'Language_Tasks'

abbrevs = df_all[(df_all['domains']==domain) & (df_all['assessment']=='Child Measures')]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']=='Child Measures') & 
                      (df_all['domains']==domain) &
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)

## Child Measures - Cognitive Testing

In [60]:
domain = 'Cognitive_Testing'

abbrevs = df_all[df_all['domains']==domain]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']=='Child Measures') & 
                      (df_all['domains']==domain) &
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


## Child Measures - Interview_of_Emotional_and_Psychological_Function

In [61]:
domain = 'Interview_of_Emotional_and_Psychological_Function'
assessment = 'Child Measures'

abbrevs = df_all[(df_all['domains']==domain) & (df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['domains']==domain) &
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


## Child Measures - Interview_of_Emotional_and_Psychological_Function

In [62]:
domain = 'Interview_of_Emotional_and_Psychological_Function'
assessment = 'Child Measures'

abbrevs = df_all[(df_all['domains']==domain) & (df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['domains']==domain) &
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


## Child Measures - Medical_Status_Measures

In [63]:
domain = 'Medical_Status_Measures'
assessment = 'Child Measures'

abbrevs = df_all[(df_all['domains']==domain) & (df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['domains']==domain) &
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


## Child Measures - Physical_Fitness_and_Status

In [64]:
domain = 'Physical_Fitness_and_Status'
assessment = 'Child Measures'

abbrevs = df_all[(df_all['domains']==domain) & (df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['domains']==domain) &
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


## Child Measures - Questionnaire_Measures_of_Emotional_and_Cognitive_Status

In [65]:
domain = 'Questionnaire_Measures_of_Emotional_and_Cognitive_Status'
assessment = 'Child Measures'

abbrevs = df_all[(df_all['domains']==domain) & (df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['domains']==domain) &
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


## Child Measures - Questionnaire_Measures_of_Family_Structure_Stress_and_Trauma

In [66]:
domain = 'Questionnaire_Measures_of_Family_Structure_Stress_and_Trauma'
assessment = 'Child Measures'

abbrevs = df_all[(df_all['domains']==domain) & (df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['domains']==domain) &
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


## Child Measures - Physiology/Motor/Neurology

In [67]:
assessment = 'Child Measures'

cols = ['Neurologic_Function', 'Physiologic_Function', 'Vision', 'Motor_Skills']

abbrevs = df_all[(df_all['domains'].isin(cols)) & (df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


## Child Measures - Questionnaire_Measures_of_Substance_Use_&_Addiction

In [68]:
domain = 'Questionnaire_Measures_of_Substance_Use_&_Addiction'
assessment = 'Child Measures'

abbrevs = df_all[(df_all['domains']==domain) & (df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['domains']==domain) &
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


## Parent Measures - Demographic_Questionnaire_Measures

In [69]:
domain = 'Demographic_Questionnaire_Measures'
assessment = 'Parent Measures'

abbrevs = df_all[(df_all['domains']==domain) & (df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['domains']==domain) &
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


## Parent Measures - Interview_of_Emotional_and_Psychological_Function

In [70]:
domain = 'Interview_of_Emotional_and_Psychological_Function'
assessment = 'Parent Measures'

abbrevs = df_all[(df_all['domains']==domain) & (df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['domains']==domain) &
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


## Parent Measures - Questionnaire_Measures_of_Family_structure_Stress_and_Trauma

In [71]:
domain = 'Questionnaire_Measures_of_Family_structure_Stress_and_Trauma'
assessment = 'Parent Measures'

abbrevs = df_all[(df_all['domains']==domain) & (df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['domains']==domain) &
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


In [72]:
assessment = 'Teacher Measures'

abbrevs = df_all[(df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['abbrevs']==abbrev)]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)


## Children's global assessment scale

In [85]:

assessment = 'Clinical Measures'

abbrevs = df_all[(df_all['assessment']==assessment)]['abbrevs'].unique()

for abbrev in abbrevs:
    
    df1 = df_all[(df_all['target']=='DX_01_Cat_new_binarize') & 
                      (df_all['assessment']==assessment) & 
                      (df_all['abbrevs']=='CGAS')]


    visualize.predictive_modeling_group(df=df1, x='participant_group', y='roc_auc_score', title=abbrev)
